<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/seq2seq/seq2seq_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/gauravd12345/language_models/blob/main/seq2seq/seq2seq_attention.ipynb)


In [ ]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

!pip install sentencepiece
import sentencepiece as spm

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


device: cuda


In [ ]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in range(len(enc)):
  enc[i] = enc[i].lower()
  dec[i] = dec[i].lower()

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

are you telling me the truth?                      | est-ce que tu me dis la vérité ?
please find a solution to the problem.             | prière de trouver une solution à ce problème.
he entertained us with jokes all evening.          | il nous a divertis avec des blagues toute la soirée.
several children are playing on the sandy beach.   | plusieurs enfants jouent sur la plage de sable.
we're not soldiers.                                | nous ne sommes pas soldats.
it's october the third.                            | nous sommes le trois octobre.
i was just wondering if you have been able to find a place to live. | je me demandais juste si tu as été capable de te trouver un endroit pour vivre.
this is all a misunderstanding.                    | tout est un malentendu.
we're taking over.                                 | nous prenons le contrôle.
you're acting like a small child.                  | tu te comportes comme un petit enfant.


In [ ]:
with open("bpe_train.txt", "w", encoding="utf-8") as f:
    for s in enc:
        f.write(str(s).lower() + "\n")
    for s in dec:
        f.write(str(s).lower() + "\n")

spm.SentencePieceTrainer.train(
    input="bpe_train.txt",
    model_prefix="bpe",
    vocab_size=8000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("bpe.model")

True

In [ ]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 512 # context vector length
max_decoder_seq = 50
batch_size = 128

lr = 0.001
epochs = 10

pad_tok = sp.pad_id()
unk_tok = sp.unk_id()
sos_tok = sp.bos_id()
eos_tok = sp.eos_id()

vocab_len = sp.get_piece_size()
enc_vocab_len = vocab_len
dec_vocab_len = vocab_len

In [ ]:
E = []
for sentence in enc:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    E.append(torch.tensor(ids, dtype=torch.long))

D = []
for sentence in dec:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    D.append(torch.tensor(ids, dtype=torch.long))

enc_pad_tok = pad_tok
dec_pad_tok = pad_tok

In [ ]:
class Seq2SeqAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.num_layers = 2

        self.enc_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=pad_tok)
        self.dec_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=dec_pad_tok)

        self.encoder = nn.GRU(
            embedding_dim,
            hidden_size,
            num_layers=self.num_layers,
            dropout=0.2,
            batch_first=True
        )

        self.decoder = nn.GRU(
            embedding_dim + hidden_size,
            hidden_size,
            num_layers=self.num_layers,
            dropout=0.2,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_len)

    def attention(self, H, s_top, e):
        # H: [batch, src_len, hidden_size]
        # s_top: [batch, hidden_size]
        # e: [batch, src_len]

        E_ij = torch.bmm(H, s_top.unsqueeze(2)).squeeze(2)
        # [batch, src_len]

        src_mask = e != pad_tok
        E_ij = E_ij.masked_fill(~src_mask, -1e9)

        wei = torch.softmax(E_ij, dim=1)

        C = torch.sum(wei.unsqueeze(2) * H, dim=1)
        # [batch, hidden_size]

        return C, wei

    def forward(self, e, d=None):
        batch_size = e.size(0)

        s0 = torch.zeros(
            self.num_layers,
            batch_size,
            hidden_size,
            device=e.device
        )

        enc_w = self.enc_embed(e)
        H, s = self.encoder(enc_w, s0)
        # H: [batch, src_len, hidden_size]
        # s: [num_layers, batch, hidden_size]

        y = []

        if d is None:
            w_i = torch.full(
                (batch_size, 1),
                sos_tok,
                dtype=torch.long,
                device=e.device
            )

            w_i = self.dec_embed(w_i)

            for i in range(max_decoder_seq):
                s_top = s[-1]
                # [batch, hidden_size]

                C, wei = self.attention(H, s_top, e)

                decoder_input = torch.cat(
                    [w_i, C.unsqueeze(1)],
                    dim=2
                )

                out, s = self.decoder(decoder_input, s)

                out = self.fc(out)

                if i < 3:
                    out[:, :, eos_tok] = -1e9

                pred = torch.argmax(out, dim=2)
                # [batch, 1]

                y.append(pred)

                w_i = self.dec_embed(pred)

            return torch.cat(y, dim=1)

        else:
            dec_w = self.dec_embed(d)

            for i in range(d.size(1)):
                s_top = s[-1]
                # [batch, hidden_size]

                C, wei = self.attention(H, s_top, e)

                decoder_input = torch.cat(
                    [dec_w[:, i:i+1, :], C.unsqueeze(1)],
                    dim=2
                )

                out, s = self.decoder(decoder_input, s)

                out = self.fc(out)

                y.append(out)

            return torch.cat(y, dim=1)

In [ ]:
class Seq2SeqDataset(Dataset):
  def __init__(self, e, d):
    self.e = e
    self.d = d

  def __getitem__(self, index):
     return self.e[index], self.d[index]

  def __len__(self):
    return len(self.e)

""" collate function for batching """
def collate_fn(batch):
  e, d = zip(*batch)

  max_e = max(len(seq) for seq in e)
  max_d = max(len(seq) for seq in d)

  pad_e = torch.zeros(len(e), max_e, dtype=torch.long)
  pad_d = torch.zeros(len(d), max_d, dtype=torch.long)

  for i in range(len(e)):
    pad_e[i] = torch.cat([e[i], torch.tensor([enc_pad_tok] * (max_e - len(e[i])))])

  for i in range(len(d)):
    pad_d[i] = torch.cat([d[i], torch.tensor([dec_pad_tok] * (max_d - len(d[i])))])

  return pad_e, pad_d

In [ ]:
seq2seq_attention = Seq2SeqAttention().to(device)

dataset = Seq2SeqDataset(E, D)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

criterion = nn.CrossEntropyLoss(ignore_index=dec_pad_tok)

optimizer = optim.Adam(
    seq2seq_attention.parameters(),
    lr=lr
)

seq2seq_attention.train()

for epoch in range(epochs):
    total_loss = 0.0

    for e, d in dataloader:
        e = e.to(device)
        d = d.to(device)

        optimizer.zero_grad()

        d_in = d[:, :-1]
        d_out = d[:, 1:]

        out = seq2seq_attention(e, d_in)

        loss = criterion(
            out.reshape(-1, out.size(-1)),
            d_out.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    print(f"Epoch: {epoch+1}/{epochs} | loss: {avg_loss:.4f}")

Epoch: 1/10 | loss: 3.0786
Epoch: 2/10 | loss: 1.8335
Epoch: 3/10 | loss: 1.4142
Epoch: 4/10 | loss: 1.1924
Epoch: 5/10 | loss: 1.0541
Epoch: 6/10 | loss: 0.9587
Epoch: 7/10 | loss: 0.8886
Epoch: 8/10 | loss: 0.8371
Epoch: 9/10 | loss: 0.7960
Epoch: 10/10 | loss: 0.7659


In [ ]:
seq2seq_attention.eval()

text = """
Yesterday I went to the market with my sister. We bought fresh bread, fruit, and some cheese for dinner.
The weather was cold, but the streets were full of people. After lunch, we visited a small bookstore near the river and
spent almost an hour looking at old novels. I was tired when I got home, but I still watched a movie before going to sleep.
"""

text = sent_tokenize(text)

for test in text:
    seq = sp.encode(test.lower(), out_type=int)
    seq = [sos_tok] + seq + [eos_tok]

    with torch.no_grad():
        seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

        pred = seq2seq_attention(seq)

        pred_ids = []

        for pred_id in pred[0].tolist():
            if pred_id == eos_tok:
                break

            if pred_id not in [sos_tok, dec_pad_tok, pad_tok]:
                pred_ids.append(pred_id)

        print(sp.decode(pred_ids))

après avoir rendu le marché, mon frère va.
nous avons acheté du pain, frais et de l'apprent gauf pour déjeuner.
la météo, la plupart des victimes est né médiévérés.
après le déjeuner, nous sommes un petit sourire et la salade, un chemin de dos à une mijile de tom et demie.
j'étais fatigué quand je suis rentré à un restaurant avant de manger tandis.
